# S&P 500 Multi-Factor Model — Backtest

End-to-end run of the strategy: build the score from trailing Rank-IC factor weights,
backtest the 25-name book behind both de-risking overlays, and compare against the
cap-weighted S&P 500.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src import factors, plots, settings
from src.optimize import Optimizer

CACHE = Path.cwd().parent / "data" / "cache"
START, END = "2001-01-01", "2025-12-31"

print(f"{len(factors.ALL_FACTORS)} live factors across {len(factors.FAMILIES)} families")
print({fam: len(flist) for fam, (flist, _) in factors.FAMILIES.items()})
print(f"IC window {settings.RANK_IC_WINDOW_MONTHS}mo | {settings.SELECTION_N} names | "
      f"{settings.TARGET_VOL:.0%} vol target | {settings.FLAT_COST_BPS}bps")

## 1. Load panels and build the score

`Optimizer.__init__` loads the cached panels, computes sector-neutral percentiles, and
derives the Rank-IC weight schedule once. `rescore()` then applies that schedule.

In [ ]:
opt = Optimizer(start=START, end=END, holdout_last_fold=False)
scored = opt.rescore()

fam_sched, _ = opt.ic_schedule
print(f"{len(fam_sched)} rebalance dates")
(fam_sched.groupby(fam_sched.index.year).mean() * 100).round(1).tail(10)

Weights are not fitted — each factor's weight is its own trailing IC, floored at zero,
with family shares bounded to [5%, 60%]. Every row above sums to 100%.

In [ ]:
lo, hi = settings.FAMILY_WEIGHT_BOUNDS
assert abs(fam_sched.sum(axis=1) - 1.0).max() < 1e-9
assert fam_sched.min().min() >= lo - 1e-9 and fam_sched.max().max() <= hi + 1e-9

rng = pd.DataFrame({"min": fam_sched.min(), "max": fam_sched.max(), "mean": fam_sched.mean()})
(rng * 100).round(1)

## 2. Backtest

Sizing parameters are the tuned configuration exported by `src/optimize.py`.

In [ ]:
summary = json.load(open(CACHE / "ic12_final_summary.json"))
sizing = {k: ([tuple(t) for t in v] if k == "drawdown_tiers" else v)
          for k, v in summary["sizing_params"].items()}

portfolio = opt.run_backtest(scored, opt.start, opt.end, sizing)
metrics = opt.metrics(portfolio)
sizing

## 3. Results

In [ ]:
crsp = opt._crsp.loc[opt._crsp["date"].between(opt.start, opt.end)]
strat, bench = plots.generate_all(portfolio, crsp)

yrs = (strat.index[-1] - strat.index[0]).days / 365.25
b_cagr = bench.iloc[-1] ** (1 / yrs) - 1
b_dd = (bench / bench.cummax() - 1).min()

pd.DataFrame({
    "Strategy": {
        "CAGR": metrics["annualized_return"],
        "Volatility": metrics["annualized_volatility"],
        "Sharpe": metrics["net_sharpe"],
        "Sortino": metrics["sortino_ratio"],
        "Calmar": metrics["calmar_ratio"],
        "Max drawdown": metrics["max_drawdown"],
        "Turnover / yr": metrics["turnover"] / yrs,
    },
    "S&P 500": {
        "CAGR": b_cagr,
        "Volatility": bench.pct_change().std() * (252 ** 0.5),
        "Sharpe": float("nan"),
        "Sortino": float("nan"),
        "Calmar": b_cagr / abs(b_dd),
        "Max drawdown": b_dd,
        "Turnover / yr": float("nan"),
    },
}).round(4)

The strategy wins on Calmar and max drawdown and loses on CAGR and Sharpe: a
lower-return, lower-risk profile rather than an outperforming one.

Figures are written to `results/performance/`.

In [ ]:
from IPython.display import Image, display

for name in ["equity_curve", "drawdown", "strategy_vs_sp500", "crisis_2008", "crisis_covid"]:
    display(Image(filename=str(Path.cwd().parent / "results" / "performance" / f"{name}.png")))